In [1]:
# --- Pratinjau 4 kelas: 1 gambar per grup (beberapa bbox digrid di kanvas 640) ---
# Kelas dari data.yaml: 0 atrophic_scar, 1 closed_comedo, 3 melasma, 6 open_comedo, 7 other
# Grup: Comedo=[1,6], Atrophic scar=[0], Other=[7], Melasma=[3]

import yaml
import numpy as np
from pathlib import Path
import cv2

SRC_BASE = Path(r"C:\Dokumen\SKRIPSI\Bimbingan\SPLIT DATA ROBUST\SPLIT DATA\Data Train")
SRC_IMAGES = SRC_BASE / "train" / "images"
SRC_LABELS = SRC_BASE / "train" / "labels"
if not SRC_IMAGES.is_dir():
    SRC_IMAGES = SRC_BASE / "images"
    SRC_LABELS = SRC_BASE / "labels"

OUT_DIR = SRC_BASE / "preview_4kelas_grid"
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(SRC_BASE / "data.yaml", "r", encoding="utf-8") as f:
    _y = yaml.safe_load(f)
CLASS_NAMES = _y.get("names", [])
if isinstance(CLASS_NAMES, dict):
    CLASS_NAMES = [CLASS_NAMES[i] for i in sorted(CLASS_NAMES.keys())]

IMG_SIZE = 640
GRID_ROWS, GRID_COLS = 3, 3
GRID_GAP = 10
N_MAX = GRID_ROWS * GRID_COLS
PADDING_PX = 20
MAX_CELL_W = (IMG_SIZE - GRID_GAP * (GRID_COLS + 1)) // GRID_COLS
MAX_CELL_H = (IMG_SIZE - GRID_GAP * (GRID_ROWS + 1)) // GRID_ROWS

GROUPS = [
    ("comedo", {1, 6}),
    ("atrophic_scar", {0}),
    ("other", {7}),
    ("melasma", {3}),
]


def get_image_path(stem, images_dir):
    for ext in (".jpg", ".jpeg", ".png"):
        p = images_dir / (stem + ext)
        if p.exists():
            return p
    return None


def parse_yolo_labels(label_path):
    rows = []
    if not label_path.exists():
        return rows
    with open(label_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            try:
                cid = int(parts[0])
                xc, yc, w, h = map(float, parts[1:5])
                rows.append((cid, xc, yc, w, h))
            except (ValueError, IndexError):
                continue
    return rows


def extract_crop_with_bbox(img, cx_px, cy_px, w_px, h_px, crop_w, crop_h, class_id, fill=128):
    h_img, w_img = img.shape[:2]
    crop_x0 = int(cx_px - crop_w / 2)
    crop_y0 = int(cy_px - crop_h / 2)
    pad_left = max(0, -crop_x0)
    pad_top = max(0, -crop_y0)
    src_x0, src_y0 = max(0, crop_x0), max(0, crop_y0)
    src_x1 = min(w_img, crop_x0 + crop_w)
    src_y1 = min(h_img, crop_y0 + crop_h)
    patch = img[src_y0:src_y1, src_x0:src_x1]
    canvas = np.full((crop_h, crop_w, 3), fill, dtype=img.dtype)
    ph, pw = patch.shape[:2]
    canvas[pad_top : pad_top + ph, pad_left : pad_left + pw] = patch
    ccx = cx_px - crop_x0 + pad_left
    ccy = cy_px - crop_y0 + pad_top
    return canvas, (
        class_id,
        ccx / crop_w,
        ccy / crop_h,
        w_px / crop_w,
        h_px / crop_h,
    )


def gather_instances(pairs, class_ids, limit):
    out = []
    for img_path, _, rows in pairs:
        for cid, xc, yc, w, h in rows:
            if cid in class_ids and len(out) < limit:
                out.append((img_path, cid, xc, yc, w, h))
            if len(out) >= limit:
                return out
    return out


pairs = []
for lp in sorted(SRC_LABELS.glob("*.txt")):
    ip = get_image_path(lp.stem, SRC_IMAGES)
    if ip is None:
        continue
    rows = parse_yolo_labels(lp)
    if rows:
        pairs.append((ip, lp, rows))

print(f"Sumber: {SRC_IMAGES} | pasangan valid: {len(pairs)}")
print(f"Output (4 file): {OUT_DIR}")

for stem_name, cids in GROUPS:
    inst = gather_instances(pairs, cids, N_MAX)
    if not inst:
        print(f"[skip] {stem_name}: tidak ada bbox untuk {cids}")
        continue
    max_w_px = max_h_px = 0
    for img_path, _, xc, yc, w, h in inst:
        img0 = cv2.imread(str(img_path))
        if img0 is None:
            continue
        H0, W0 = img0.shape[:2]
        max_w_px = max(max_w_px, w * W0)
        max_h_px = max(max_h_px, h * H0)
    crop_w = min(int(max_w_px) + 2 * PADDING_PX, MAX_CELL_W)
    crop_h = min(int(max_h_px) + 2 * PADDING_PX, MAX_CELL_H)
    crop_w, crop_h = max(crop_w, 32), max(crop_h, 32)

    canvas = np.full((IMG_SIZE, IMG_SIZE, 3), 128, dtype=np.uint8)
    labels_640 = []
    for i, (img_path, cid, xc, yc, w, h) in enumerate(inst):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        hi, wi = img.shape[:2]
        cx_px, cy_px = xc * wi, yc * hi
        w_px, h_px = w * wi, h * hi
        crop_img, bbox_norm = extract_crop_with_bbox(
            img, cx_px, cy_px, w_px, h_px, crop_w, crop_h, class_id=cid
        )
        row, col = i // GRID_COLS, i % GRID_COLS
        ox = GRID_GAP + col * (crop_w + GRID_GAP)
        oy = GRID_GAP + row * (crop_h + GRID_GAP)
        if crop_img.shape[1] != crop_w or crop_img.shape[0] != crop_h:
            crop_img = cv2.resize(crop_img, (crop_w, crop_h))
        canvas[oy : oy + crop_h, ox : ox + crop_w] = crop_img
        xc_c, yc_c, wc, hc = bbox_norm[1], bbox_norm[2], bbox_norm[3], bbox_norm[4]
        cx_640 = ox + xc_c * crop_w
        cy_640 = oy + yc_c * crop_h
        w_640, h_640 = wc * crop_w, hc * crop_h
        labels_640.append(
            (cid, cx_640 / IMG_SIZE, cy_640 / IMG_SIZE, w_640 / IMG_SIZE, h_640 / IMG_SIZE)
        )

    vis = canvas.copy()
    hh, ww = vis.shape[:2]
    for cid, xc, yc, bw, bh in labels_640:
        x1 = int((xc - bw / 2) * ww)
        y1 = int((yc - bh / 2) * hh)
        x2 = int((xc + bw / 2) * ww)
        y2 = int((yc + bh / 2) * hh)
        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
        nm = CLASS_NAMES[cid] if cid < len(CLASS_NAMES) else str(cid)
        cv2.putText(vis, nm, (x1, max(12, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 1)

    out_path = OUT_DIR / f"{stem_name}_grid640.jpg"
    cv2.imwrite(str(out_path), vis)
    print(f"  -> {out_path.name} | bbox: {len(labels_640)} | kelas id: {sorted(cids)}")

print("Selesai: 4 gambar (atau lebih sedikit jika suatu kelas kosong).")


Sumber: C:\Dokumen\SKRIPSI\Bimbingan\SPLIT DATA ROBUST\SPLIT DATA\Data Train\images | pasangan valid: 170
Output (4 file): C:\Dokumen\SKRIPSI\Bimbingan\SPLIT DATA ROBUST\SPLIT DATA\Data Train\preview_4kelas_grid
  -> comedo_grid640.jpg | bbox: 9 | kelas id: [1, 6]
  -> atrophic_scar_grid640.jpg | bbox: 9 | kelas id: [0]
  -> other_grid640.jpg | bbox: 9 | kelas id: [7]
  -> melasma_grid640.jpg | bbox: 9 | kelas id: [3]
Selesai: 4 gambar (atau lebih sedikit jika suatu kelas kosong).


In [ ]:
import os
import yaml
import numpy as np
from pathlib import Path
from tqdm import tqdm
import cv2

# Sumber data: train_patches_640_whole_remapped (gambar 640x640)
SRC_BASE = Path(r"C:\Dokumen\SKRIPSI\Bimbingan\SPLIT DATA ROBUST\SPLIT DATA\Data Train")
SRC_IMAGES = SRC_BASE / "train" / "images"
SRC_LABELS = SRC_BASE / "train" / "labels"
if not SRC_IMAGES.is_dir():
    SRC_IMAGES = SRC_BASE / "images"
    SRC_LABELS = SRC_BASE / "labels"

# Output: folder baru — 4 kelas minoritas (bukan hanya nodule)
OUT_BASE = Path(r"C:\Dokumen\SKRIPSI\Bimbingan\SPLIT DATA ROBUST\SPLIT DATA\Data Train\AUGMENTASI_DATA_MINORITAS_V1\MINORITAS_4KELAS_640_EKSTRAK")
OUT_IMAGES = OUT_BASE / "images"
OUT_LABELS = OUT_BASE / "labels"
OUT_VIS = OUT_BASE / "visualisasi"
OUT_IMAGES.mkdir(parents=True, exist_ok=True)
OUT_LABELS.mkdir(parents=True, exist_ok=True)
OUT_VIS.mkdir(parents=True, exist_ok=True)
print(f"Sumber: {SRC_IMAGES}")
print(f"Output: {OUT_BASE}")

# Load class names dari data.yaml sumber
SRC_DATA_YAML = SRC_BASE / "data.yaml"
with open(SRC_DATA_YAML, "r", encoding="utf-8") as f:
    src_yaml = yaml.safe_load(f)
CLASS_NAMES = src_yaml.get("names", [])
if isinstance(CLASS_NAMES, dict):
    CLASS_NAMES = [CLASS_NAMES[i] for i in sorted(CLASS_NAMES.keys())]

# Target: Hypertrophic scar (2), Nevus (4), Papule (7), Pustule (8)
# WAJIB pakai label format 9 kelas (remap). Jika sumber masih 10 kelas, ubah ID atau pakai folder remapped.
TARGET_CLASS_IDS = {2, 4, 7, 8}
TARGET_NAMES = {2: "hypertrophic_scar", 4: "nevus", 7: "papule", 8: "pustule"}
print("Kelas yang diekstrak (cek konsistensi dengan data.yaml):")
for cid in sorted(TARGET_CLASS_IDS):
    nm = CLASS_NAMES[cid] if cid < len(CLASS_NAMES) else "?"
    print(f"  id={cid} -> yaml: {nm} | expected: {TARGET_NAMES.get(cid)}")

def get_image_path(stem, images_dir):
    for ext in (".jpg", ".jpeg", ".png"):
        p = images_dir / (stem + ext)
        if p.exists():
            return p
    return None

def parse_yolo_labels(label_path):
    rows = []
    if not label_path or not Path(label_path).exists():
        return rows
    with open(label_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            try:
                cid = int(parts[0])
                xc, yc, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                rows.append((cid, xc, yc, w, h))
            except (ValueError, IndexError):
                continue
    return rows

# Pasangan (image_path, label_path, rows) dari sumber
label_files = list(SRC_LABELS.glob("*.txt"))
image_label_pairs = []
for lp in tqdm(label_files, desc="Load image-label pairs"):
    img_path = get_image_path(lp.stem, SRC_IMAGES)
    if img_path is None:
        continue
    rows = parse_yolo_labels(lp)
    if not rows:
        continue
    image_label_pairs.append((img_path, lp, rows))

print(f"Pasangan image-label valid: {len(image_label_pairs)}")
for cid in sorted(TARGET_CLASS_IDS):
    n_c = sum(1 for _, _, rows in image_label_pairs for r in rows if r[0] == cid)
    print(f"  Total bbox {TARGET_NAMES.get(cid, cid)} (id={cid}) di sumber: {n_c}")
total_target_src = sum(1 for _, _, rows in image_label_pairs for r in rows if r[0] in TARGET_CLASS_IDS)
print(f"Total bbox (4 kelas gabungan) di sumber: {total_target_src}")

Sumber: C:\Dokumen\SKRIPSI\Bimbingan\SPLIT DATA ROBUST\SPLIT DATA\Data Train\images
Output: C:\Dokumen\SKRIPSI\Bimbingan\SPLIT DATA ROBUST\SPLIT DATA\Data Train\AUGMENTASI_DATA_MINORITAS_V1\MINORITAS_4KELAS_640_EKSTRAK
Kelas yang diekstrak (cek konsistensi dengan data.yaml):
  id=2 -> yaml: hypertrophic_scar | expected: hypertrophic_scar
  id=4 -> yaml: nevus | expected: nevus
  id=7 -> yaml: other | expected: papule
  id=8 -> yaml: papule | expected: pustule


Load image-label pairs:   0%|          | 0/165 [00:00<?, ?it/s]

Load image-label pairs: 100%|██████████| 165/165 [00:03<00:00, 47.67it/s]


Pasangan image-label valid: 165
  Total bbox hypertrophic_scar (id=2) di sumber: 494
  Total bbox nevus (id=4) di sumber: 845
  Total bbox papule (id=7) di sumber: 243
  Total bbox pustule (id=8) di sumber: 3161
Total bbox (4 kelas gabungan) di sumber: 4743


In [22]:
# --- Grid kanvas 640x640 (sesuai kode) ---
IMG_SIZE = 640
PADDING_PX = 20
GRID_GAP = 10
GRID_ROWS, GRID_COLS = 3, 3
MAX_CELL_W = (IMG_SIZE - GRID_GAP * (GRID_COLS + 1)) // GRID_COLS  # ~206
MAX_CELL_H = (IMG_SIZE - GRID_GAP * (GRID_ROWS + 1)) // GRID_ROWS

# Ukuran crop: bbox terbesar di antara 4 kelas target + padding (gambar sumber 640x640)
max_w_px = max_h_px = 0
for img_path, lp, rows in image_label_pairs:
    for cid, xc, yc, w, h in rows:
        if cid not in TARGET_CLASS_IDS:
            continue
        w_px = w * IMG_SIZE
        h_px = h * IMG_SIZE
        max_w_px = max(max_w_px, w_px)
        max_h_px = max(max_h_px, h_px)

crop_w = min(int(max_w_px) + 2 * PADDING_PX, MAX_CELL_W)
crop_h = min(int(max_h_px) + 2 * PADDING_PX, MAX_CELL_H)
print(f"Bbox terbesar 4 kelas (px): w={max_w_px:.0f}, h={max_h_px:.0f}")
print(f"Crop standar (padding {PADDING_PX}px): {crop_w}x{crop_h} (max cell {MAX_CELL_W}x{MAX_CELL_H})")

Bbox terbesar 4 kelas (px): w=274, h=85
Crop standar (padding 20px): 200x125 (max cell 200x200)


In [23]:
# --- Ekstraksi crop per objek (4 kelas) + label (bbox dalam koordinat crop, tidak bergeser) ---
# Setiap crop: (crop_img, bbox normalized dalam crop), stem, idx
def extract_crop_with_bbox(img, cx_px, cy_px, w_px, h_px, crop_w, crop_h, class_id, fill=0):
    h_img, w_img = img.shape[:2]
    crop_x0 = int(cx_px - crop_w / 2)
    crop_y0 = int(cy_px - crop_h / 2)
    pad_left = max(0, -crop_x0)
    pad_top = max(0, -crop_y0)
    src_x0 = max(0, crop_x0)
    src_y0 = max(0, crop_y0)
    src_x1 = min(w_img, crop_x0 + crop_w)
    src_y1 = min(h_img, crop_y0 + crop_h)
    patch = img[src_y0:src_y1, src_x0:src_x1]
    canvas = np.full((crop_h, crop_w, 3), fill, dtype=img.dtype)
    dy = pad_top
    dx = pad_left
    ph, pw = patch.shape[:2]
    canvas[dy:dy+ph, dx:dx+pw] = patch
    # Bbox objek dalam crop (normalized): center & size relatif ke crop (exact, tidak bergeser)
    canvas_cx = cx_px - crop_x0 + pad_left
    canvas_cy = cy_px - crop_y0 + pad_top
    xc_norm = canvas_cx / crop_w
    yc_norm = canvas_cy / crop_h
    w_norm = w_px / crop_w
    h_norm = h_px / crop_h
    return canvas, (class_id, xc_norm, yc_norm, w_norm, h_norm)

crops_with_bbox = []  # (crop_img, (cid, xc, yc, w, h)_norm, img_stem, idx)
for img_path, lp, rows in tqdm(image_label_pairs, desc="Ekstraksi crop 4 kelas"):
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    h_img, w_img = img.shape[:2]
    stem = img_path.stem
    idx = 0
    for cid, xc, yc, w, h in rows:
        if cid not in TARGET_CLASS_IDS:
            continue
        cx_px = xc * w_img
        cy_px = yc * h_img
        w_px = w * w_img
        h_px = h * h_img
        crop_img, bbox_norm = extract_crop_with_bbox(
            img, cx_px, cy_px, w_px, h_px, crop_w, crop_h, class_id=cid
        )
        crops_with_bbox.append((crop_img, bbox_norm, stem, idx))
        idx += 1

print(f"Total crop (4 kelas) diekstrak: {len(crops_with_bbox)}")

Ekstraksi crop 4 kelas: 100%|██████████| 165/165 [00:19<00:00,  8.68it/s]

Total crop (4 kelas) diekstrak: 4743


In [24]:
def write_yolo_label(out_path, rows):
    with open(out_path, "w", encoding="utf-8") as f:
        for cid, xc, yc, w, h in rows:
            f.write(f"{cid} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")

def draw_bboxes(img, rows, color_target=(0,255,0), thickness=2):
    h, w = img.shape[:2]
    out = img.copy()
    for cid, xc, yc, bw, bh in rows:
        x1 = int((xc - bw/2) * w)
        y1 = int((yc - bh/2) * h)
        x2 = int((xc + bw/2) * w)
        y2 = int((yc + bh/2) * h)
        color = color_target if cid in TARGET_CLASS_IDS else (255,165,0)
        cv2.rectangle(out, (x1,y1), (x2,y2), color, thickness)
        name = CLASS_NAMES[cid] if cid < len(CLASS_NAMES) else str(cid)
        cv2.putText(out, name, (x1, y1-4), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
    return out

In [25]:
# --- Komposisi ke kanvas 640 + simpan ke folder images, labels, visualisasi ---
# Label tidak bergeser: offset kanvas (offset_x, offset_y) + koordinat dalam crop (xc_c*crop_w, ...)
BACKGROUND_PATH = Path(r"C:\Dokumen\SKRIPSI\Bimbingan\ACNE_DETECTION_ORI\train_patches_640_whole\train\images\4_jpg.rf.5746b806de6057b3b07062497e226dc8_x1280_y320.jpg")
bg = cv2.imread(str(BACKGROUND_PATH))
if bg is None:
    bg = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    bg[:] = (128, 128, 128)
else:
    bg = cv2.resize(bg, (IMG_SIZE, IMG_SIZE))

def place_crops_on_canvas(crop_list, canvas, crop_w, crop_h):
    """Letakkan crop di grid. Bbox di kanvas = offset + (xc_c*crop_w, yc_c*crop_h) → tidak bergeser."""
    out = canvas.copy()
    labels_640 = []
    n_per_grid = GRID_ROWS * GRID_COLS
    for i in range(0, min(len(crop_list), n_per_grid)):
        crop_img, bbox_norm, stem, _ = crop_list[i]
        cid, xc_c, yc_c, w_c, h_c = bbox_norm
        row, col = i // GRID_COLS, i % GRID_COLS
        offset_x = GRID_GAP + col * (crop_w + GRID_GAP)
        offset_y = GRID_GAP + row * (crop_h + GRID_GAP)
        if crop_img.shape[1] != crop_w or crop_img.shape[0] != crop_h:
            crop_img = cv2.resize(crop_img, (crop_w, crop_h))
        out[offset_y:offset_y+crop_h, offset_x:offset_x+crop_w] = crop_img
        # Koordinat kanvas 640: exact (tidak bergeser)
        cx_640 = offset_x + xc_c * crop_w
        cy_640 = offset_y + yc_c * crop_h
        w_640 = w_c * crop_w
        h_640 = h_c * crop_h
        labels_640.append((cid, cx_640 / IMG_SIZE, cy_640 / IMG_SIZE, w_640 / IMG_SIZE, h_640 / IMG_SIZE))
    return out, labels_640

n_composite = (len(crops_with_bbox) + GRID_ROWS * GRID_COLS - 1) // (GRID_ROWS * GRID_COLS)
for k in tqdm(range(n_composite), desc="Komposisi kanvas"):
    start = k * (GRID_ROWS * GRID_COLS)
    batch = crops_with_bbox[start:start + GRID_ROWS * GRID_COLS]
    canvas_out, labels_640 = place_crops_on_canvas(batch, bg, crop_w, crop_h)
    stem_out = f"min4_composite_{k:05d}"
    cv2.imwrite(str(OUT_IMAGES / (stem_out + ".jpg")), canvas_out)
    write_yolo_label(OUT_LABELS / (stem_out + ".txt"), labels_640)
    vis_out = draw_bboxes(canvas_out, labels_640)
    cv2.imwrite(str(OUT_VIS / (stem_out + ".jpg")), vis_out)

yaml_out = {"path": str(OUT_BASE.absolute()), "train": "images", "val": "", "nc": len(CLASS_NAMES), "names": CLASS_NAMES}
with open(OUT_BASE / "data.yaml", "w", encoding="utf-8") as f:
    yaml.dump(yaml_out, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print(f"Gambar komposit: {OUT_IMAGES} ({n_composite} file)")
print(f"Label: {OUT_LABELS}")
print(f"Visualisasi: {OUT_VIS}")
print(f"data.yaml: {OUT_BASE / 'data.yaml'}")

# --- Ringkasan bbox per kelas ---
cnt = {}
for _, bbox_norm, _, _ in crops_with_bbox:
    cid = bbox_norm[0]
    cnt[cid] = cnt.get(cid, 0) + 1
print("\n" + "=" * 50)
print("TOTAL CROP PER KELAS (4 kelas target):")
for cid in sorted(TARGET_CLASS_IDS):
    print(f"  {TARGET_NAMES.get(cid)} (id={cid}): {cnt.get(cid, 0)}")
print(f"TOTAL (semua): {len(crops_with_bbox)}")
print("=" * 50)

Komposisi kanvas: 100%|██████████| 527/527 [00:05<00:00, 97.20it/s] 

Gambar komposit: C:\Dokumen\SKRIPSI\Bimbingan\SPLIT DATA ROBUST\SPLIT DATA\Data Train\AUGMENTASI_DATA_MINORITAS_V1\MINORITAS_4KELAS_640_EKSTRAK\images (527 file)
Label: C:\Dokumen\SKRIPSI\Bimbingan\SPLIT DATA ROBUST\SPLIT DATA\Data Train\AUGMENTASI_DATA_MINORITAS_V1\MINORITAS_4KELAS_640_EKSTRAK\labels
Visualisasi: C:\Dokumen\SKRIPSI\Bimbingan\SPLIT DATA ROBUST\SPLIT DATA\Data Train\AUGMENTASI_DATA_MINORITAS_V1\MINORITAS_4KELAS_640_EKSTRAK\visualisasi
data.yaml: C:\Dokumen\SKRIPSI\Bimbingan\SPLIT DATA ROBUST\SPLIT DATA\Data Train\AUGMENTASI_DATA_MINORITAS_V1\MINORITAS_4KELAS_640_EKSTRAK\data.yaml

TOTAL CROP PER KELAS (4 kelas target):
  hypertrophic_scar (id=2): 494
  nevus (id=4): 845
  papule (id=7): 243
  pustule (id=8): 3161
TOTAL (semua): 4743


In [26]:
# ========== Augmentasi: Flip horizontal (dari crop 4 kelas, label tidak bergeser) ==========
OUT_BASE_FLIP = OUT_BASE.parent / "MINORITAS_4KELAS_640_AUG_flip_horizontal"
OUT_IMAGES_FLIP = OUT_BASE_FLIP / "images"
OUT_LABELS_FLIP = OUT_BASE_FLIP / "labels"
OUT_VIS_FLIP = OUT_BASE_FLIP / "visualisasi"
OUT_IMAGES_FLIP.mkdir(parents=True, exist_ok=True)
OUT_LABELS_FLIP.mkdir(parents=True, exist_ok=True)
OUT_VIS_FLIP.mkdir(parents=True, exist_ok=True)

# Flip H: gambar dibalik, bbox normalized: xc_baru = 1 - xc (center flip horizontal)
crops_flip_h = []
for crop_img, (cid, xc, yc, w, h), stem, idx in crops_with_bbox:
    flipped = cv2.flip(crop_img, 1)
    crops_flip_h.append((flipped, (cid, 1.0 - xc, yc, w, h), stem, idx))

n_comp_flip = (len(crops_flip_h) + GRID_ROWS * GRID_COLS - 1) // (GRID_ROWS * GRID_COLS)
for k in tqdm(range(n_comp_flip), desc="Kanvas flip horizontal"):
    start = k * (GRID_ROWS * GRID_COLS)
    batch = crops_flip_h[start:start + GRID_ROWS * GRID_COLS]
    canvas_out, labels_640 = place_crops_on_canvas(batch, bg, crop_w, crop_h)
    name = f"min4_flipH_{k:05d}"
    cv2.imwrite(str(OUT_IMAGES_FLIP / (name + ".jpg")), canvas_out)
    write_yolo_label(OUT_LABELS_FLIP / (name + ".txt"), labels_640)
    vis_out = draw_bboxes(canvas_out, labels_640)
    cv2.imwrite(str(OUT_VIS_FLIP / (name + ".jpg")), vis_out)

yaml_flip = {"path": str(OUT_BASE_FLIP.absolute()), "train": "images", "val": "", "nc": len(CLASS_NAMES), "names": CLASS_NAMES}
with open(OUT_BASE_FLIP / "data.yaml", "w", encoding="utf-8") as f:
    yaml.dump(yaml_flip, f, default_flow_style=False, allow_unicode=True, sort_keys=False)
print(f"Flip H: {OUT_BASE_FLIP} | Kanvas: {n_comp_flip} | images/labels/visualisasi + data.yaml")

Kanvas flip horizontal: 100%|██████████| 527/527 [00:05<00:00, 99.32it/s] 

Flip H: C:\Dokumen\SKRIPSI\Bimbingan\SPLIT DATA ROBUST\SPLIT DATA\Data Train\AUGMENTASI_DATA_MINORITAS_V1\MINORITAS_4KELAS_640_AUG_flip_horizontal | Kanvas: 527 | images/labels/visualisasi + data.yaml


In [27]:
# ========== Augmentasi: Flip vertikal (dari crop 4 kelas, label tidak bergeser) ==========
OUT_BASE_FLIPV = OUT_BASE.parent / "MINORITAS_4KELAS_640_AUG_flip_vertical"
OUT_IMAGES_FLIPV = OUT_BASE_FLIPV / "images"
OUT_LABELS_FLIPV = OUT_BASE_FLIPV / "labels"
OUT_VIS_FLIPV = OUT_BASE_FLIPV / "visualisasi"
OUT_IMAGES_FLIPV.mkdir(parents=True, exist_ok=True)
OUT_LABELS_FLIPV.mkdir(parents=True, exist_ok=True)
OUT_VIS_FLIPV.mkdir(parents=True, exist_ok=True)

# Flip V: cv2.flip(..., 0); bbox normalized: yc_baru = 1 - yc
crops_flip_v = []
for crop_img, (cid, xc, yc, w, h), stem, idx in crops_with_bbox:
    flipped = cv2.flip(crop_img, 0)
    crops_flip_v.append((flipped, (cid, xc, 1.0 - yc, w, h), stem, idx))

n_comp_fv = (len(crops_flip_v) + GRID_ROWS * GRID_COLS - 1) // (GRID_ROWS * GRID_COLS)
for k in tqdm(range(n_comp_fv), desc="Kanvas flip vertikal"):
    start = k * (GRID_ROWS * GRID_COLS)
    batch = crops_flip_v[start:start + GRID_ROWS * GRID_COLS]
    canvas_out, labels_640 = place_crops_on_canvas(batch, bg, crop_w, crop_h)
    name = f"min4_flipV_{k:05d}"
    cv2.imwrite(str(OUT_IMAGES_FLIPV / (name + ".jpg")), canvas_out)
    write_yolo_label(OUT_LABELS_FLIPV / (name + ".txt"), labels_640)
    vis_out = draw_bboxes(canvas_out, labels_640)
    cv2.imwrite(str(OUT_VIS_FLIPV / (name + ".jpg")), vis_out)

yaml_fv = {"path": str(OUT_BASE_FLIPV.absolute()), "train": "images", "val": "", "nc": len(CLASS_NAMES), "names": CLASS_NAMES}
with open(OUT_BASE_FLIPV / "data.yaml", "w", encoding="utf-8") as f:
    yaml.dump(yaml_fv, f, default_flow_style=False, allow_unicode=True, sort_keys=False)
print(f"Flip V: {OUT_BASE_FLIPV} | Kanvas: {n_comp_fv} | images/labels/visualisasi + data.yaml")

Kanvas flip vertikal: 100%|██████████| 527/527 [00:06<00:00, 87.17it/s] 

Flip V: C:\Dokumen\SKRIPSI\Bimbingan\SPLIT DATA ROBUST\SPLIT DATA\Data Train\AUGMENTASI_DATA_MINORITAS_V1\MINORITAS_4KELAS_640_AUG_flip_vertical | Kanvas: 527 | images/labels/visualisasi + data.yaml


In [28]:
# ========== Augmentasi: Rotasi 32° (dari crop 4 kelas, bbox ikut rotasi) ==========
ROTATE_DEG = 32

def rotate_patch_and_bbox(patch, obj_x, obj_y, obj_w, obj_h, bg_canvas, angle_deg=ROTATE_DEG):
    """Rotate patch by angle_deg around center. Padding = crop dari bg_canvas. Return (rotated_patch, new_obj_x, new_obj_y, new_obj_w, new_obj_h)."""
    h, w = patch.shape[:2]
    center = (w / 2.0, h / 2.0)
    M = cv2.getRotationMatrix2D(center, angle_deg, 1.0)
    corners_patch = np.array([[0, 0], [w, 0], [w, h], [0, h]], dtype=np.float32)
    pts = cv2.transform(corners_patch.reshape(1, -1, 2), M).reshape(-1, 2)
    x_min, y_min = pts.min(axis=0)
    x_max, y_max = pts.max(axis=0)
    new_w = int(np.ceil(x_max - x_min))
    new_h = int(np.ceil(y_max - y_min))
    M[0, 2] += -x_min
    M[1, 2] += -y_min
    bh, bw = bg_canvas.shape[:2]
    if new_w <= bw and new_h <= bh:
        x0 = max(0, (bw - new_w) // 2)
        y0 = max(0, (bh - new_h) // 2)
        canvas_crop = bg_canvas[y0:y0 + new_h, x0:x0 + new_w].copy()
    else:
        canvas_crop = cv2.resize(bg_canvas, (new_w, new_h))
    warped = cv2.warpAffine(patch, M, (new_w, new_h), borderMode=cv2.BORDER_CONSTANT)
    mask_ones = np.ones((h, w), dtype=np.uint8) * 255
    warped_mask = cv2.warpAffine(mask_ones, M, (new_w, new_h), borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    warped_mask_3 = np.expand_dims(warped_mask, axis=2)
    rotated = np.where(warped_mask_3 > 0, warped, canvas_crop).astype(np.uint8)
    corners_bbox = np.array([
        [obj_x, obj_y], [obj_x + obj_w, obj_y],
        [obj_x + obj_w, obj_y + obj_h], [obj_x, obj_y + obj_h]
    ], dtype=np.float32)
    pts_bbox = cv2.transform(corners_bbox.reshape(1, -1, 2), M).reshape(-1, 2)
    xmin_n = max(0, pts_bbox[:, 0].min())
    ymin_n = max(0, pts_bbox[:, 1].min())
    xmax_n = min(new_w, pts_bbox[:, 0].max())
    ymax_n = min(new_h, pts_bbox[:, 1].max())
    if xmax_n <= xmin_n or ymax_n <= ymin_n:
        return rotated, 0, 0, 1, 1
    new_obj_x = xmin_n
    new_obj_y = ymin_n
    new_obj_w = xmax_n - xmin_n
    new_obj_h = ymax_n - ymin_n
    return rotated, new_obj_x, new_obj_y, new_obj_w, new_obj_h

OUT_BASE_ROT = OUT_BASE.parent / "MINORITAS_4KELAS_640_AUG_rotate32"
OUT_IMAGES_ROT = OUT_BASE_ROT / "images"
OUT_LABELS_ROT = OUT_BASE_ROT / "labels"
OUT_VIS_ROT = OUT_BASE_ROT / "visualisasi"
OUT_IMAGES_ROT.mkdir(parents=True, exist_ok=True)
OUT_LABELS_ROT.mkdir(parents=True, exist_ok=True)
OUT_VIS_ROT.mkdir(parents=True, exist_ok=True)

# Konversi crops_with_bbox (normalized) ke pixel untuk rotasi, lalu resize hasil ke (crop_w, crop_h) + bbox norm baru
crops_rot = []
for crop_img, (cid, xc_n, yc_n, w_n, h_n), stem, idx in tqdm(crops_with_bbox, desc="Rotasi 32°"):
    ph, pw = crop_img.shape[:2]
    obj_x = (xc_n - w_n / 2) * pw
    obj_y = (yc_n - h_n / 2) * ph
    obj_w = w_n * pw
    obj_h = h_n * ph
    rot_patch, nox, noy, nw, nh = rotate_patch_and_bbox(crop_img, obj_x, obj_y, obj_w, obj_h, bg)
    rw, rh = rot_patch.shape[1], rot_patch.shape[0]
    resized = cv2.resize(rot_patch, (crop_w, crop_h))
    xc_new = (nox + nw / 2) / rw
    yc_new = (noy + nh / 2) / rh
    w_new = nw / rw
    h_new = nh / rh
    crops_rot.append((resized, (cid, xc_new, yc_new, w_new, h_new), stem, idx))

n_comp_rot = (len(crops_rot) + GRID_ROWS * GRID_COLS - 1) // (GRID_ROWS * GRID_COLS)
for k in tqdm(range(n_comp_rot), desc="Kanvas rotasi 32°"):
    start = k * (GRID_ROWS * GRID_COLS)
    batch = crops_rot[start:start + GRID_ROWS * GRID_COLS]
    canvas_out, labels_640 = place_crops_on_canvas(batch, bg, crop_w, crop_h)
    name = f"min4_rot32_{k:05d}"
    cv2.imwrite(str(OUT_IMAGES_ROT / (name + ".jpg")), canvas_out)
    write_yolo_label(OUT_LABELS_ROT / (name + ".txt"), labels_640)
    vis_out = draw_bboxes(canvas_out, labels_640)
    cv2.imwrite(str(OUT_VIS_ROT / (name + ".jpg")), vis_out)

yaml_rot = {"path": str(OUT_BASE_ROT.absolute()), "train": "images", "val": "", "nc": len(CLASS_NAMES), "names": CLASS_NAMES}
with open(OUT_BASE_ROT / "data.yaml", "w", encoding="utf-8") as f:
    yaml.dump(yaml_rot, f, default_flow_style=False, allow_unicode=True, sort_keys=False)
print(f"Rotasi 32°: {OUT_BASE_ROT} | Kanvas: {n_comp_rot} | images/labels/visualisasi + data.yaml")

Kanvas rotasi 32°: 100%|██████████| 527/527 [00:05<00:00, 95.46it/s] 

Rotasi 32°: C:\Dokumen\SKRIPSI\Bimbingan\SPLIT DATA ROBUST\SPLIT DATA\Data Train\AUGMENTASI_DATA_MINORITAS_V1\MINORITAS_4KELAS_640_AUG_rotate32 | Kanvas: 527 | images/labels/visualisasi + data.yaml


In [29]:
# ========== Augmentasi: Contrast (dan Contrast + Saturation) dari crop 4 kelas ==========
CONTRAST_AUG = 1.8   # 1.0 = tanpa perubahan
SATURATION_AUG = 1.3 # 1.0 = tanpa perubahan

def apply_contrast(img_bgr, factor):
    out = img_bgr.astype(np.float32) * factor
    return np.clip(out, 0, 255).astype(np.uint8)

def apply_saturation(img_bgr, factor):
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    hsv[:, :, 1] = np.clip(hsv[:, :, 1].astype(np.float32) * factor, 0, 255).astype(np.uint8)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

# ---- 1) Contrast saja (bbox tidak berubah) ----
OUT_BASE_CONTRAST = OUT_BASE.parent / "MINORITAS_4KELAS_640_AUG_contrast"
OUT_IMAGES_CONTRAST = OUT_BASE_CONTRAST / "images"
OUT_LABELS_CONTRAST = OUT_BASE_CONTRAST / "labels"
OUT_VIS_CONTRAST = OUT_BASE_CONTRAST / "visualisasi"
OUT_IMAGES_CONTRAST.mkdir(parents=True, exist_ok=True)
OUT_LABELS_CONTRAST.mkdir(parents=True, exist_ok=True)
OUT_VIS_CONTRAST.mkdir(parents=True, exist_ok=True)

crops_contrast = []
for crop_img, bbox_norm, stem, idx in tqdm(crops_with_bbox, desc="Contrast"):
    out = apply_contrast(crop_img, CONTRAST_AUG)
    crops_contrast.append((out, bbox_norm, stem, idx))

n_comp_ct = (len(crops_contrast) + GRID_ROWS * GRID_COLS - 1) // (GRID_ROWS * GRID_COLS)
for k in tqdm(range(n_comp_ct), desc="Kanvas contrast"):
    start = k * (GRID_ROWS * GRID_COLS)
    batch = crops_contrast[start:start + GRID_ROWS * GRID_COLS]
    canvas_out, labels_640 = place_crops_on_canvas(batch, bg, crop_w, crop_h)
    name = f"min4_contrast_{k:05d}"
    cv2.imwrite(str(OUT_IMAGES_CONTRAST / (name + ".jpg")), canvas_out)
    write_yolo_label(OUT_LABELS_CONTRAST / (name + ".txt"), labels_640)
    vis_out = draw_bboxes(canvas_out, labels_640)
    cv2.imwrite(str(OUT_VIS_CONTRAST / (name + ".jpg")), vis_out)

yaml_ct = {"path": str(OUT_BASE_CONTRAST.absolute()), "train": "images", "val": "", "nc": len(CLASS_NAMES), "names": CLASS_NAMES}
with open(OUT_BASE_CONTRAST / "data.yaml", "w", encoding="utf-8") as f:
    yaml.dump(yaml_ct, f, default_flow_style=False, allow_unicode=True, sort_keys=False)
print(f"Contrast: {OUT_BASE_CONTRAST} | Kanvas: {n_comp_ct}")

# ---- 2) Contrast + Saturation ----
OUT_BASE_CS = OUT_BASE.parent / "MINORITAS_4KELAS_640_AUG_contrast_saturation"
OUT_IMAGES_CS = OUT_BASE_CS / "images"
OUT_LABELS_CS = OUT_BASE_CS / "labels"
OUT_VIS_CS = OUT_BASE_CS / "visualisasi"
OUT_IMAGES_CS.mkdir(parents=True, exist_ok=True)
OUT_LABELS_CS.mkdir(parents=True, exist_ok=True)
OUT_VIS_CS.mkdir(parents=True, exist_ok=True)

crops_cs = []
for crop_img, bbox_norm, stem, idx in tqdm(crops_with_bbox, desc="Contrast + Saturasi"):
    out = apply_contrast(crop_img, CONTRAST_AUG)
    out = apply_saturation(out, SATURATION_AUG)
    crops_cs.append((out, bbox_norm, stem, idx))

n_comp_cs = (len(crops_cs) + GRID_ROWS * GRID_COLS - 1) // (GRID_ROWS * GRID_COLS)
for k in tqdm(range(n_comp_cs), desc="Kanvas contrast+saturasi"):
    start = k * (GRID_ROWS * GRID_COLS)
    batch = crops_cs[start:start + GRID_ROWS * GRID_COLS]
    canvas_out, labels_640 = place_crops_on_canvas(batch, bg, crop_w, crop_h)
    name = f"min4_cs_{k:05d}"
    cv2.imwrite(str(OUT_IMAGES_CS / (name + ".jpg")), canvas_out)
    write_yolo_label(OUT_LABELS_CS / (name + ".txt"), labels_640)
    vis_out = draw_bboxes(canvas_out, labels_640)
    cv2.imwrite(str(OUT_VIS_CS / (name + ".jpg")), vis_out)

yaml_cs = {"path": str(OUT_BASE_CS.absolute()), "train": "images", "val": "", "nc": len(CLASS_NAMES), "names": CLASS_NAMES}
with open(OUT_BASE_CS / "data.yaml", "w", encoding="utf-8") as f:
    yaml.dump(yaml_cs, f, default_flow_style=False, allow_unicode=True, sort_keys=False)
print(f"Contrast+Saturation: {OUT_BASE_CS} | Kanvas: {n_comp_cs}")
print(f"  Semua augmentasi disimpan di: {OUT_BASE.parent} (images / labels / visualisasi + data.yaml)")

Kanvas contrast: 100%|██████████| 527/527 [00:05<00:00, 90.59it/s] 


Contrast: C:\Dokumen\SKRIPSI\Bimbingan\SPLIT DATA ROBUST\SPLIT DATA\Data Train\AUGMENTASI_DATA_MINORITAS_V1\MINORITAS_4KELAS_640_AUG_contrast | Kanvas: 527


Kanvas contrast+saturasi: 100%|██████████| 527/527 [00:06<00:00, 87.26it/s] 

Contrast+Saturation: C:\Dokumen\SKRIPSI\Bimbingan\SPLIT DATA ROBUST\SPLIT DATA\Data Train\AUGMENTASI_DATA_MINORITAS_V1\MINORITAS_4KELAS_640_AUG_contrast_saturation | Kanvas: 527
  Semua augmentasi disimpan di: C:\Dokumen\SKRIPSI\Bimbingan\SPLIT DATA ROBUST\SPLIT DATA\Data Train\AUGMENTASI_DATA_MINORITAS_V1 (images / labels / visualisasi + data.yaml)


In [30]:
from pathlib import Path

BASE = Path(r"C:\Dokumen\SKRIPSI\Bimbingan\SPLIT DATA ROBUST\SPLIT DATA\Data Train")
AUGMENTASI_V1 = BASE / "AUGMENTASI_DATA_MINORITAS_V1"
# Sama dengan cell 0: kelas target (9 kelas)
TARGET_CLASS_IDS = {2, 4, 7, 8}
TARGET_NAMES = {2: "hypertrophic_scar", 4: "nevus", 7: "papule", 8: "pustule"}

FOLDERS = [
    AUGMENTASI_V1 / "MINORITAS_4KELAS_640_EKSTRAK",
    AUGMENTASI_V1 / "MINORITAS_4KELAS_640_AUG_flip_horizontal",
    AUGMENTASI_V1 / "MINORITAS_4KELAS_640_AUG_flip_vertical",
    AUGMENTASI_V1 / "MINORITAS_4KELAS_640_AUG_rotate32",
    AUGMENTASI_V1 / "MINORITAS_4KELAS_640_AUG_contrast",
    AUGMENTASI_V1 / "MINORITAS_4KELAS_640_AUG_contrast_saturation",
    BASE / "DATA TRAIN YANG BELUM BALANCE",
]


def count_target_classes_in_labels(labels_dir):
    """Hitung jumlah baris per class_id untuk TARGET_CLASS_IDS."""
    if not labels_dir or not labels_dir.exists():
        return None
    per_cid = {cid: 0 for cid in TARGET_CLASS_IDS}
    for f in labels_dir.glob("*.txt"):
        with open(f, "r", encoding="utf-8") as fp:
            for line in fp:
                parts = line.strip().split()
                if len(parts) >= 5 and parts[0].isdigit():
                    cid = int(parts[0])
                    if cid in per_cid:
                        per_cid[cid] += 1
    return per_cid


print("Jumlah label per kelas (2,4,7,8) per folder:\n")
for folder in FOLDERS:
    labels_dir = folder / "labels"
    if not labels_dir.exists():
        labels_dir = folder / "train" / "labels"
    counts = count_target_classes_in_labels(labels_dir)
    name = folder.name
    if counts is None:
        print(f"{name}: (folder labels tidak ada)")
        continue
    tot = sum(counts.values())
    detail = ", ".join(f"{TARGET_NAMES[c]}={counts[c]}" for c in sorted(TARGET_CLASS_IDS))
    print(f"{name}")
    print(f"  Total bbox (4 kelas): {tot}  |  {detail}")
    print()

Jumlah label per kelas (2,4,7,8) per folder:

MINORITAS_4KELAS_640_EKSTRAK
  Total bbox (4 kelas): 4743  |  hypertrophic_scar=494, nevus=845, papule=243, pustule=3161

MINORITAS_4KELAS_640_AUG_flip_horizontal
  Total bbox (4 kelas): 4743  |  hypertrophic_scar=494, nevus=845, papule=243, pustule=3161

MINORITAS_4KELAS_640_AUG_flip_vertical
  Total bbox (4 kelas): 4743  |  hypertrophic_scar=494, nevus=845, papule=243, pustule=3161

MINORITAS_4KELAS_640_AUG_rotate32
  Total bbox (4 kelas): 4743  |  hypertrophic_scar=494, nevus=845, papule=243, pustule=3161

MINORITAS_4KELAS_640_AUG_contrast
  Total bbox (4 kelas): 4743  |  hypertrophic_scar=494, nevus=845, papule=243, pustule=3161

MINORITAS_4KELAS_640_AUG_contrast_saturation
  Total bbox (4 kelas): 4743  |  hypertrophic_scar=494, nevus=845, papule=243, pustule=3161

DATA TRAIN YANG BELUM BALANCE: (folder labels tidak ada)
